In [ ]:
# ---- 0. Install (Kaggle already has torch/numpy/matplotlib) ----
!pip install -q "monai>=1.3" nibabel scikit-image plotly gradio onnx onnxruntime tqdm joblib pandas nilearn

In [ ]:
# ---- 1. Imports & global configuration ----
import os, re, sys, json, time, copy, glob, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}" +
      (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))


class CFG:
    # ---------------- run mode ----------------
    QUICK_MODE = False            # True -> 40 cases, 3 epochs each (pipeline sanity check)
    SEED = 42

    # ---------------- paths ----------------
    DATA_ROOT = None              # auto-detected if None (folder that contains BraTS20_Training_XXX)
    WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./neuroassist_out")
    CACHE_DIR = WORK_DIR / "cache"
    CKPT_DIR = WORK_DIR / "checkpoints"
    FIG_DIR = WORK_DIR / "figures"
    OUT_DIR = WORK_DIR / "outputs"

    # ---------------- data ----------------
    MODALITIES = ["flair", "t1", "t1ce", "t2"]
    CROP = (slice(56, 184), slice(56, 184), slice(13, 141))   # 240x240x155 -> 128^3 (brain-centred)
    VOL_SHAPE = (128, 128, 128)
    PATCH = (96, 96, 96)          # training patch (inference runs on the full 128^3 crop)
    VAL_FRACTION = 0.20
    N_CASES = None                # use only first N cases (debug)
    BATCH_SIZE = 2
    NUM_WORKERS = min(4, os.cpu_count() or 1)
    USE_ELASTIC = True            # elastic deformation augmentation (low probability)

    # ---------------- models ----------------
    TEACHER_FILTERS = (32, 64, 128, 256, 320)   # nnU-Net-style, residual blocks (the "SOTA-style" baseline)
    STUDENT_FILTERS = (8, 16, 32, 64, 128)      # lightweight
    # ---------------- training ----------------
    TEACHER_EPOCHS = 50
    STUDENT_EPOCHS = 50
    LR = 2e-4
    KD_ALPHA = 1.0                # weight of distillation loss
    KD_TEMP = 2.0                 # distillation temperature
    VAL_EVERY = 2
    USE_AMP = True
    RUN_SCRATCH_ABLATION = False  # also train the student WITHOUT distillation (shows KD benefit)

    # ---------------- extensions (all heavy ones are OFF by default) ----------------
    LONG_RUN = False              # True -> 150 epochs each ("train longer")
    RUN_CV = False                # 5-fold cross-validation of the lightweight model
    N_FOLDS = 5
    CV_EPOCHS = 30
    CV_WITH_KD = False            # also train a fold-specific teacher (5x teacher cost!)
    RUN_FEATURE_KD = False        # feature-level distillation variant
    FEAT_LAMBDA = 1.0
    RUN_REGION_HEAD = False       # sigmoid WT/TC/ET head variant
    RUN_DWSEP = False             # depthwise-separable student variant
    STUDENT_DW_FILTERS = (12, 24, 48, 96, 160)
    RUN_MC_DROPOUT_STUDENT = False  # student trained with dropout, for MC-dropout uncertainty
    STUDENT_DROPOUT = 0.1
    MC_SAMPLES = 10


if CFG.LONG_RUN:
    CFG.TEACHER_EPOCHS = CFG.STUDENT_EPOCHS = 150
if CFG.QUICK_MODE:
    CFG.N_CASES, CFG.TEACHER_EPOCHS, CFG.STUDENT_EPOCHS, CFG.VAL_EVERY = 40, 3, 3, 1
    CFG.CV_EPOCHS = 2

for d in (CFG.CACHE_DIR, CFG.CKPT_DIR, CFG.FIG_DIR, CFG.OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=CFG.SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()

# Region definitions (labels AFTER remapping 4 -> 3)
REGIONS = {"ET": (3,), "TC": (1, 3), "WT": (1, 2, 3)}
LABEL_NAMES = {1: "Necrotic / non-enhancing core", 2: "Peritumoral edema", 3: "Enhancing tumor"}
LABEL_COLORS = {1: (0.92, 0.18, 0.18), 2: (0.15, 0.80, 0.45), 3: (1.00, 0.85, 0.10)}   # red / green / yellow

In [ ]:
import nibabel as nib
from joblib import Parallel, delayed


def scan_for_cases(base):
    """Walk `base` and return the first directory that directly contains BraTS20_Training_XXX folders."""
    if not base or not os.path.isdir(base):
        return None
    for dp, dn, _ in os.walk(base):
        if any(d.startswith("BraTS20_Training_") for d in dn):
            return dp
    return None


def find_data_root():
    """Robust against Kaggle's different mount layouts (/kaggle/input/<name>/... or /kaggle/input/datasets/<owner>/<name>/...)
    and against a DATA_ROOT that points one level too high (e.g. .../BraTS2020_TrainingData)."""
    cands = [CFG.DATA_ROOT,
             "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation",
             "/kaggle/input/brats20-dataset-training-validation"]
    for c in cands:
        hit = scan_for_cases(c)
        if hit:
            return hit
    for base in ["/kaggle/input", "/content", "/data", ".", str(CFG.WORK_DIR)]:
        hit = scan_for_cases(base)
        if hit:
            return hit
    try:  # Colab / local fallback (needs Kaggle credentials)
        import kagglehub
        hit = scan_for_cases(kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation"))
        if hit:
            return hit
    except Exception as e:
        print("kagglehub fallback failed:", e)
    raise FileNotFoundError("BraTS2020 not found. Add the Kaggle dataset or set CFG.DATA_ROOT to any folder above the cases.")


def find_case_files(case_dir):
    """Map modality -> file path (robust to the odd seg filename of case 355 in the Kaggle copy)."""
    files = {}
    for f in sorted(os.listdir(case_dir)):
        fl = f.lower()
        if not (fl.endswith(".nii") or fl.endswith(".nii.gz")):
            continue
        m = re.search(r"(flair|t1ce|t1gd|t1|t2)\.nii", fl)
        if m:
            files["t1ce" if m.group(1) == "t1gd" else m.group(1)] = os.path.join(case_dir, f)
        elif "seg" in fl:
            files["seg"] = os.path.join(case_dir, f)
    return files


DATA_ROOT = find_data_root()
ALL_IDS = sorted(d for d in os.listdir(DATA_ROOT)
                 if d.startswith("BraTS20_Training_") and os.path.isdir(os.path.join(DATA_ROOT, d)))
assert len(ALL_IDS) > 0, f"No BraTS20_Training_XXX folders in {DATA_ROOT}"
if CFG.N_CASES:
    ALL_IDS = ALL_IDS[:CFG.N_CASES]
print(f"Data root: {DATA_ROOT}\nCases found: {len(ALL_IDS)}")

In [ ]:
def load_nii(path):
    return nib.load(path).get_fdata(dtype=np.float32)


def zscore_nonzero(vol):
    """Clip to robust range, then z-score using non-zero (brain) voxels only; background stays 0."""
    mask = vol != 0
    if mask.sum() == 0:
        return vol
    v = vol[mask]
    lo, hi = np.percentile(v, [0.5, 99.5])
    v = np.clip(v, lo, hi)
    out = np.zeros_like(vol, dtype=np.float32)
    out[mask] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def preprocess_modalities(vols):
    """vols: dict modality -> (240,240,155) array  ->  (4,128,128,128) float32 (z-scored, cropped)."""
    chans = [zscore_nonzero(vols[m])[CFG.CROP] for m in CFG.MODALITIES]
    return np.stack(chans).astype(np.float32)


def preprocess_label(seg):
    seg = np.rint(seg).astype(np.uint8)[CFG.CROP]
    seg[seg == 4] = 3            # <-- label remap 4 -> 3 (contiguous classes 0,1,2,3)
    return seg


def cache_case(cid):
    ip, lp = CFG.CACHE_DIR / f"{cid}_img.npy", CFG.CACHE_DIR / f"{cid}_lbl.npy"
    if ip.exists() and lp.exists():
        return cid
    files = find_case_files(os.path.join(DATA_ROOT, cid))
    img = preprocess_modalities({m: load_nii(files[m]) for m in CFG.MODALITIES})
    lbl = preprocess_label(load_nii(files["seg"]))
    np.save(ip, img.astype(np.float16)); np.save(lp, lbl)
    return cid


def load_cached(cid):
    img = np.load(CFG.CACHE_DIR / f"{cid}_img.npy").astype(np.float32)
    lbl = np.load(CFG.CACHE_DIR / f"{cid}_lbl.npy")
    return img, lbl


t0 = time.time()
_ = Parallel(n_jobs=min(4, os.cpu_count() or 1))(delayed(cache_case)(c) for c in tqdm(ALL_IDS, desc="Caching"))
print(f"Cached {len(ALL_IDS)} cases in {time.time()-t0:.0f}s -> {CFG.CACHE_DIR}")

# Sanity checks
img0, lbl0 = load_cached(ALL_IDS[0])
print("image:", img0.shape, img0.dtype, "| label:", lbl0.shape, "| label values:", np.unique(lbl0))
assert set(np.unique(lbl0)).issubset({0, 1, 2, 3}), "label remap failed"

In [ ]:
# ---- Patient-level train / validation split ----
rng = np.random.RandomState(CFG.SEED)
perm = rng.permutation(len(ALL_IDS))
n_val = max(4, int(round(CFG.VAL_FRACTION * len(ALL_IDS))))
VAL_IDS = sorted(ALL_IDS[i] for i in perm[:n_val])
TRAIN_IDS = sorted(ALL_IDS[i] for i in perm[n_val:])
json.dump({"train": TRAIN_IDS, "val": VAL_IDS}, open(CFG.OUT_DIR / "split.json", "w"), indent=1)
print(f"train: {len(TRAIN_IDS)} cases | val: {len(VAL_IDS)} cases")

In [ ]:
from monai.transforms import (Compose, EnsureTyped, RandCropByPosNegLabeld, RandFlipd, RandRotate90d,
                              RandAffined, Rand3DElasticd, RandScaleIntensityd, RandShiftIntensityd,
                              RandGaussianNoised)


def get_train_transform():
    keys = ["image", "label"]
    tf = [
        RandCropByPosNegLabeld(keys, label_key="label", spatial_size=CFG.PATCH, pos=3, neg=1, num_samples=1),
        RandFlipd(keys, prob=0.5, spatial_axis=0),
        RandFlipd(keys, prob=0.5, spatial_axis=1),
        RandFlipd(keys, prob=0.5, spatial_axis=2),
        RandRotate90d(keys, prob=0.3, max_k=3, spatial_axes=(0, 1)),
        RandAffined(keys, prob=0.25, rotate_range=(0.2, 0.2, 0.2), scale_range=(0.1, 0.1, 0.1),
                    mode=("bilinear", "nearest"), padding_mode="zeros"),
    ]
    if CFG.USE_ELASTIC:
        tf.append(Rand3DElasticd(keys, sigma_range=(5, 8), magnitude_range=(50, 100), prob=0.1,
                                 mode=("bilinear", "nearest"), padding_mode="zeros"))
    tf += [
        RandScaleIntensityd("image", factors=0.1, prob=0.5),
        RandShiftIntensityd("image", offsets=0.1, prob=0.5),
        RandGaussianNoised("image", prob=0.2, std=0.05),
        EnsureTyped(keys, dtype=torch.float32, track_meta=False),
    ]
    return Compose(tf)


def get_val_transform():
    return Compose([EnsureTyped(["image", "label"], dtype=torch.float32, track_meta=False)])


class BraTSCached(torch.utils.data.Dataset):
    def __init__(self, ids, transform):
        self.ids, self.transform = ids, transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img, lbl = load_cached(self.ids[i])
        out = self.transform({"image": img, "label": lbl[None].astype(np.float32)})
        return out[0] if isinstance(out, (list, tuple)) else out     # RandCropByPosNegLabeld returns a list


def _worker_init(worker_id):
    """Give every dataloader worker a different augmentation RNG stream."""
    seed = torch.initial_seed() % (2 ** 32)
    ds = torch.utils.data.get_worker_info().dataset
    ds.transform.set_random_state(seed=seed)
    np.random.seed(seed); random.seed(seed)


def make_loaders(train_ids=None, val_ids=None):
    tr = BraTSCached(train_ids or TRAIN_IDS, get_train_transform())
    va = BraTSCached(val_ids or VAL_IDS, get_val_transform())
    nw = CFG.NUM_WORKERS
    kw = dict(num_workers=nw, pin_memory=DEVICE.type == "cuda", worker_init_fn=_worker_init if nw else None)
    train_loader = torch.utils.data.DataLoader(tr, batch_size=CFG.BATCH_SIZE, shuffle=True, drop_last=True,
                                               persistent_workers=nw > 0, **kw)
    val_loader = torch.utils.data.DataLoader(va, batch_size=1, shuffle=False, **kw)
    return train_loader, val_loader


train_loader, val_loader = make_loaders()
b = next(iter(train_loader))
print("train batch:", b["image"].shape, b["label"].shape, "| label values:", torch.unique(b["label"]).tolist())

In [ ]:
AXES = {"Axial": 2, "Coronal": 1, "Sagittal": 0}


def take(vol, axis, idx):
    """Extract a 2D slice of a 3D volume and transpose for display (imshow origin='lower')."""
    return np.take(vol, int(idx), axis=axis)


def best_slice(seg, axis=2):
    other = tuple(a for a in range(3) if a != axis)
    s = (seg > 0).sum(axis=other)
    return int(np.argmax(s)) if s.max() > 0 else seg.shape[axis] // 2


def gray_norm(sl):
    nz = sl[sl != 0]
    if nz.size == 0:
        return np.zeros_like(sl, dtype=np.float32)
    lo, hi = np.percentile(nz, [1, 99])
    return np.clip((sl - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)


def overlay_rgb(base2d, seg2d=None, alpha=0.5):
    g = gray_norm(base2d)
    rgb = np.stack([g, g, g], -1)
    if seg2d is not None:
        for c, col in LABEL_COLORS.items():
            m = seg2d == c
            rgb[m] = (1 - alpha) * rgb[m] + alpha * np.array(col)
    return np.transpose(rgb, (1, 0, 2))      # (W,H,3) so that imshow(origin='lower') has the usual orientation


def draw(ax, base2d, seg2d=None, title="", alpha=0.5, dist2d=None, margins=()):
    ax.imshow(overlay_rgb(base2d, seg2d, alpha), origin="lower")
    if dist2d is not None:
        for m, col in zip(margins, ["cyan", "orange", "magenta", "white"]):
            if m > 0 and dist2d.min() <= m:
                ax.contour(dist2d.T, levels=[m], colors=[col], linewidths=1.3)
    ax.set_title(title, fontsize=10); ax.axis("off")


def legend_handles():
    from matplotlib.patches import Patch
    return [Patch(color=LABEL_COLORS[c], label=LABEL_NAMES[c]) for c in (3, 1, 2)]

In [ ]:
# ---- Fig 1: the four MRI modalities + ground truth for a training case ----
cid = TRAIN_IDS[0]
img, gt = load_cached(cid)
k = best_slice(gt, 2)
fig, axs = plt.subplots(1, 5, figsize=(20, 4.2))
for a, (m, i) in zip(axs[:4], zip(CFG.MODALITIES, range(4))):
    draw(a, take(img[i], 2, k), None, {"flair": "T2-FLAIR", "t1": "T1", "t1ce": "T1 post-contrast", "t2": "T2"}[m])
draw(axs[4], take(img[2], 2, k), take(gt, 2, k), "Ground truth on T1ce")
fig.legend(handles=legend_handles(), loc="lower center", ncol=3, frameon=False)
fig.suptitle(f"{cid} — axial slice {k}", y=1.02)
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / "fig1_modalities.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
from monai.networks.nets import DynUNet
from monai.losses import DiceCELoss


def build_model(filters, res_block, out_ch=4, dropout=None):
    n = len(filters)
    return DynUNet(spatial_dims=3, in_channels=4, out_channels=out_ch,
                   kernel_size=[3] * n, strides=[1] + [2] * (n - 1), upsample_kernel_size=[2] * (n - 1),
                   filters=list(filters), res_block=res_block, deep_supervision=False, dropout=dropout)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_teacher():
    return build_model(CFG.TEACHER_FILTERS, res_block=True)


def build_student():
    return build_model(CFG.STUDENT_FILTERS, res_block=False)


_t, _s = build_teacher(), build_student()
print(f"Baseline (teacher): {count_params(_t)/1e6:6.2f} M params")
print(f"Lightweight (student): {count_params(_s)/1e6:6.2f} M params  "
      f"(-> {count_params(_t)/count_params(_s):.1f}x fewer)")
del _t, _s

In [ ]:
seg_loss_fn = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False, batch=True)


def kd_loss_fn(s_logits, t_logits, T):
    s = F.log_softmax(s_logits / T, dim=1)
    t = F.softmax(t_logits / T, dim=1)
    return F.kl_div(s, t, reduction="none").sum(1).mean() * (T * T)


def region_dice_torch(pred, gt):
    """pred, gt: integer label volumes (same shape) -> dict region -> dice (1.0 if both empty)."""
    out = {}
    for r, ids in REGIONS.items():
        p = torch.zeros_like(pred, dtype=torch.bool); g = torch.zeros_like(gt, dtype=torch.bool)
        for i in ids:
            p |= pred == i; g |= gt == i
        s = p.sum().item() + g.sum().item()
        out[r] = 1.0 if s == 0 else 2.0 * (p & g).sum().item() / s
    return out


@torch.no_grad()
def validate(model, loader):
    model.eval()
    scores = {r: [] for r in REGIONS}
    for b in loader:
        x = b["image"].to(DEVICE, non_blocking=True)
        y = b["label"].long().to(DEVICE).squeeze(1)
        with torch.autocast(DEVICE.type, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            logits = model(x)
        pred = logits.argmax(1)
        for k in range(x.shape[0]):
            d = region_dice_torch(pred[k], y[k])
            for r in REGIONS:
                scores[r].append(d[r])
    res = {r: float(np.mean(v)) for r, v in scores.items()}
    res["mean"] = float(np.mean(list(res.values())))
    return res

In [ ]:
def train_model(name, model, train_loader, val_loader, epochs, lr=None, teacher=None,
                kd_alpha=0.0, kd_T=2.0, resume=True, feat_distill=None, feat_lambda=1.0):
    """Generic trainer. If `teacher` is given, adds the distillation loss. Returns (best model, history df)."""
    lr = lr or CFG.LR
    use_amp = CFG.USE_AMP and DEVICE.type == "cuda"
    model.to(DEVICE)
    best_p, last_p = CFG.CKPT_DIR / f"{name}_best.pt", CFG.CKPT_DIR / f"{name}_last.pt"
    params = list(model.parameters()) + (list(feat_distill.adapters.parameters()) if feat_distill is not None else [])
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    hist, best, start = [], -1.0, 0

    if resume and last_p.exists():
        ck = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
        print(f"[{name}] resumed from epoch {start}")

    if teacher is not None:
        teacher.to(DEVICE).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)

    for ep in range(start, epochs):
        model.train(); t0 = time.time(); run, n = 0.0, 0
        pbar = tqdm(train_loader, leave=False, desc=f"{name} {ep+1}/{epochs}")
        if feat_distill is not None:
            feat_distill.enabled = True
        for b in pbar:
            x = b["image"].to(DEVICE, non_blocking=True)
            y = b["label"].long().to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = seg_loss_fn(logits.float(), y)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(x)
                    loss = loss + kd_alpha * kd_loss_fn(logits.float(), t_logits.float(), kd_T)
                    if feat_distill is not None:      # feature-level distillation (encoder features)
                        loss = loss + feat_lambda * feat_distill.loss()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0)
            scaler.step(opt); scaler.update()
            run += loss.item(); n += 1
            pbar.set_postfix(loss=f"{run/n:.4f}")
        sched.step()
        if feat_distill is not None:
            feat_distill.enabled = False

        row = {"epoch": ep + 1, "train_loss": run / max(n, 1), "lr": opt.param_groups[0]["lr"],
               "time_s": time.time() - t0, "val_mean": np.nan, "val_ET": np.nan, "val_TC": np.nan, "val_WT": np.nan}
        if (ep + 1) % CFG.VAL_EVERY == 0 or ep + 1 == epochs:
            v = validate(model, val_loader)
            row.update(val_mean=v["mean"], val_ET=v["ET"], val_TC=v["TC"], val_WT=v["WT"])
            if v["mean"] > best:
                best = v["mean"]
                torch.save({"model": model.state_dict(), "epoch": ep, "val": v}, best_p)
            print(f"[{name}] ep {ep+1:3d}  loss {row['train_loss']:.4f}  "
                  f"Dice ET {v['ET']:.3f} TC {v['TC']:.3f} WT {v['WT']:.3f} | mean {v['mean']:.3f}  "
                  f"(best {best:.3f})  {row['time_s']:.0f}s")
        hist.append(row)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "hist": hist, "best": best, "epoch": ep}, last_p)

    if best_p.exists():
        model.load_state_dict(torch.load(best_p, map_location=DEVICE, weights_only=False)["model"])
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(CFG.OUT_DIR / f"history_{name}.csv", index=False)
    return model, hist_df

In [ ]:
teacher = build_teacher()
teacher, hist_teacher = train_model("teacher", teacher, train_loader, val_loader, CFG.TEACHER_EPOCHS)
print("Best baseline val Dice:", validate(teacher, val_loader))

In [ ]:
student = build_student()
student, hist_student = train_model("student_kd", student, train_loader, val_loader, CFG.STUDENT_EPOCHS,
                                    teacher=teacher, kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP)
print("Best lightweight val Dice:", validate(student, val_loader))

In [ ]:
student_scratch, hist_scratch = None, None
if CFG.RUN_SCRATCH_ABLATION:
    student_scratch = build_student()
    student_scratch, hist_scratch = train_model("student_scratch", student_scratch, train_loader, val_loader,
                                                CFG.STUDENT_EPOCHS)
    print("Student (no KD) val Dice:", validate(student_scratch, val_loader))
else:
    print("Skipped (CFG.RUN_SCRATCH_ABLATION = False)")

In [ ]:
# ---- Fig 2: training curves ----
curves = [("Baseline", hist_teacher), ("Lightweight (KD)", hist_student)]
if hist_scratch is not None:
    curves.append(("Lightweight (no KD)", hist_scratch))
fig, axs = plt.subplots(1, 3, figsize=(18, 4.2))
for name, h in curves:
    axs[0].plot(h["epoch"], h["train_loss"], label=name)
    v = h.dropna(subset=["val_mean"])
    axs[1].plot(v["epoch"], v["val_mean"], marker="o", label=name)
axs[0].set(title="Training loss", xlabel="epoch"); axs[1].set(title="Validation mean Dice (ET/TC/WT)", xlabel="epoch")
for r, c in zip(["val_ET", "val_TC", "val_WT"], ["#e6b800", "#d62728", "#2ca02c"]):
    for (name, h), ls in zip(curves[:2], ["-", "--"]):
        v = h.dropna(subset=[r]); axs[2].plot(v["epoch"], v[r], ls, color=c, label=f"{name} {r[4:]}")
axs[2].set(title="Per-region validation Dice", xlabel="epoch")
for a in axs:
    a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / "fig2_training_curves.png", dpi=130); plt.show()

In [ ]:
from scipy import ndimage as ndi


def dice_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    s = p.sum() + g.sum()
    return 1.0 if s == 0 else float(2.0 * (p & g).sum() / s)


def _surface(m):
    return m & ~ndi.binary_erosion(m, structure=ndi.generate_binary_structure(3, 1), border_value=0)


def hd95_np(p, g, spacing=1.0):
    p, g = p.astype(bool), g.astype(bool)
    if not p.any() and not g.any():
        return 0.0
    if not p.any() or not g.any():
        return np.nan
    ps, gs = _surface(p), _surface(g)
    d_to_g = ndi.distance_transform_edt(~gs, sampling=spacing)
    d_to_p = ndi.distance_transform_edt(~ps, sampling=spacing)
    return float(np.percentile(np.concatenate([d_to_g[ps], d_to_p[gs]]), 95))


@torch.no_grad()
def predict(model, img, use_amp=True):
    """img: (4,128,128,128) float32 numpy -> (uint8 label volume, latency in ms)."""
    model.eval()
    x = torch.from_numpy(img).float().unsqueeze(0).to(DEVICE)
    amp = use_amp and CFG.USE_AMP and DEVICE.type == "cuda"
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.autocast(DEVICE.type, enabled=amp):
        logits = model(x)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8), ms


def evaluate_models(models, ids):
    """models: dict name -> nn.Module. Returns per-case DataFrame."""
    for m in models.values():                           # warm-up (cuDNN autotune etc.)
        predict(m, load_cached(ids[0])[0])
    rows = []
    for cid in tqdm(ids, desc="Evaluating"):
        img, gt = load_cached(cid)
        for name, m in models.items():
            pred, ms = predict(m, img)
            row = {"case": cid, "model": name, "latency_ms": ms}
            for r, lab in REGIONS.items():
                p, g = np.isin(pred, lab), np.isin(gt, lab)
                row[f"Dice_{r}"], row[f"HD95_{r}"] = dice_np(p, g), hd95_np(p, g)
            row["Dice_NCR"] = dice_np(pred == 1, gt == 1)
            row["Dice_ED"] = dice_np(pred == 2, gt == 2)
            rows.append(row)
    df = pd.DataFrame(rows)
    df["Dice_mean"] = df[["Dice_ET", "Dice_TC", "Dice_WT"]].mean(axis=1)
    return df


MODELS = {"Baseline (teacher)": teacher, "Lightweight (KD student)": student}
if student_scratch is not None:
    MODELS["Lightweight (no KD)"] = student_scratch

per_case = evaluate_models(MODELS, VAL_IDS)
per_case.to_csv(CFG.OUT_DIR / "per_case_metrics.csv", index=False)
acc_cols = ["Dice_ET", "Dice_TC", "Dice_WT", "Dice_mean", "Dice_NCR", "Dice_ED", "HD95_ET", "HD95_TC", "HD95_WT"]
acc = per_case.groupby("model")[acc_cols].mean().round(3)     # NaN HD95 cases are skipped by pandas
acc

In [ ]:
# ---- Efficiency benchmark: params / size / FLOPs / latency / memory ----
from torch.utils.flop_counter import FlopCounterMode


def gflops(model, shape=(1, 4, 64, 64, 64)):
    """FLOPs measured at 64^3 and scaled by 8 to the 128^3 input (fully-convolutional => linear in volume)."""
    m = copy.deepcopy(model).cpu().eval()
    x = torch.randn(*shape)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    return fc.get_total_flops() * (128 / shape[-1]) ** 3 / 1e9


@torch.no_grad()
def bench(model, device, half=False, shape=(1, 4, 128, 128, 128), warm=3, runs=10):
    m = copy.deepcopy(model).to(device).eval()
    x = torch.randn(*shape, device=device)
    if half:
        m, x = m.half(), x.half()
    if device.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    for _ in range(warm):
        m(x)
    ts = []
    for _ in range(runs):
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); m(x)
        if device.type == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    mem = torch.cuda.max_memory_allocated() / 2 ** 20 if device.type == "cuda" else np.nan
    del m, x
    return float(np.mean(ts)), float(np.std(ts)), mem


eff_rows = []
for name, m in MODELS.items():
    row = {"model": name, "Params (M)": count_params(m) / 1e6,
           "Size FP32 (MB)": count_params(m) * 4 / 2 ** 20}
    try:
        row["GFLOPs (128^3)"] = gflops(m)
    except Exception as e:
        print("FLOP counting unavailable:", e); row["GFLOPs (128^3)"] = np.nan
    if DEVICE.type == "cuda":
        row["GPU FP32 (ms)"], _, row["GPU FP32 VRAM (MB)"] = bench(m, DEVICE, half=False)
        row["GPU FP16 (ms)"], _, _ = bench(m, DEVICE, half=True)
    else:
        row["GPU FP32 (ms)"] = row["GPU FP16 (ms)"] = row["GPU FP32 VRAM (MB)"] = np.nan
    row["CPU (ms)"], _, _ = bench(m, torch.device("cpu"), warm=1, runs=2)
    eff_rows.append(row)
eff = pd.DataFrame(eff_rows).set_index("model").round(2)
eff

In [ ]:
# ---- Final results table (accuracy + efficiency) ----
results = eff.join(acc[["Dice_ET", "Dice_TC", "Dice_WT", "Dice_mean", "HD95_ET", "HD95_TC", "HD95_WT"]])
base_name = "Baseline (teacher)"; stu_name = "Lightweight (KD student)"
speed = {
    "Parameter reduction": results.loc[base_name, "Params (M)"] / results.loc[stu_name, "Params (M)"],
    "FLOP reduction": results.loc[base_name, "GFLOPs (128^3)"] / results.loc[stu_name, "GFLOPs (128^3)"],
    "CPU speed-up": results.loc[base_name, "CPU (ms)"] / results.loc[stu_name, "CPU (ms)"],
    "GPU FP16 speed-up": results.loc[base_name, "GPU FP16 (ms)"] / results.loc[stu_name, "GPU FP16 (ms)"],
    "Mean-Dice drop (abs.)": results.loc[base_name, "Dice_mean"] - results.loc[stu_name, "Dice_mean"],
}
results.to_csv(CFG.OUT_DIR / "results_table.csv")
display(results.T)
print("\n".join(f"{k:24s} {v:6.2f}" + ("x" if "drop" not in k else "") for k, v in speed.items()))

In [ ]:
# ---- Fig 3: speed / size vs accuracy ----
names = list(results.index); cols = ["#4c72b0", "#dd8452", "#8c8c8c"][:len(names)]
fig, axs = plt.subplots(1, 4, figsize=(20, 4.2))
axs[0].bar(names, results["Params (M)"], color=cols); axs[0].set_title("Parameters (M)")
axs[1].bar(names, results["GFLOPs (128^3)"], color=cols); axs[1].set_title("GFLOPs per 128³ volume")
lat_col = "GPU FP16 (ms)" if DEVICE.type == "cuda" else "CPU (ms)"
axs[2].bar(names, results[lat_col], color=cols); axs[2].set_title(f"Latency — {lat_col}")
axs[3].bar(names, results["Dice_mean"], color=cols); axs[3].set_title("Mean Dice (ET/TC/WT)")
axs[3].set_ylim(max(0, results["Dice_mean"].min() - 0.1), 1)
for a in axs:
    a.tick_params(axis="x", rotation=15); a.grid(alpha=.3, axis="y")
    for p in a.patches:
        a.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / "fig3_efficiency.png", dpi=130); plt.show()

# Dice distribution per region
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for a, r in zip(axs, ["ET", "TC", "WT"]):
    data = [per_case[per_case.model == n][f"Dice_{r}"].values for n in names]
    a.boxplot(data, showmeans=True)
    a.set_xticks(range(1, len(names) + 1))
    a.set_xticklabels([n.replace(" (", "\n(") for n in names])
    a.set_title(f"Dice — {r}"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / "fig4_dice_boxplots.png", dpi=130); plt.show()

In [ ]:
# ---- Fig 5: qualitative comparison on validation patients ----
def plot_prediction_comparison(ids, models, fname):
    fig, axs = plt.subplots(len(ids), 5, figsize=(18, 3.6 * len(ids)), squeeze=False)
    for r, cid in enumerate(ids):
        img, gt = load_cached(cid)
        preds = {k: predict(m, img)[0] for k, m in models.items()}
        k = best_slice(gt, 2); base = take(img[2], 2, k)
        stu = preds[stu_name]
        draw(axs[r, 0], base, None, f"{cid}\nT1ce")
        draw(axs[r, 1], base, take(gt, 2, k), "Ground truth")
        draw(axs[r, 2], base, take(preds[base_name], 2, k), f"Baseline  (WT Dice {dice_np(preds[base_name]>0, gt>0):.3f})")
        draw(axs[r, 3], base, take(stu, 2, k), f"Lightweight (WT Dice {dice_np(stu>0, gt>0):.3f})")
        err = (take(stu, 2, k) != take(gt, 2, k))
        rgb = np.transpose(np.stack([gray_norm(base)] * 3, -1), (1, 0, 2)).copy()
        rgb[err.T] = [1, 0, 1]
        axs[r, 4].imshow(rgb, origin="lower"); axs[r, 4].set_title("Lightweight errors (magenta)"); axs[r, 4].axis("off")
    fig.legend(handles=legend_handles(), loc="lower center", ncol=3, frameon=False)
    plt.tight_layout(rect=(0, 0.03, 1, 1)); plt.savefig(CFG.FIG_DIR / fname, dpi=120, bbox_inches="tight"); plt.show()


plot_prediction_comparison(VAL_IDS[:3], {base_name: teacher, stu_name: student}, "fig5_qualitative.png")

## 8. Surgical-planning tools

Everything below operates on the segmentation (here: the lightweight model's prediction) and is shared with the demo:

* **Tumor burden** — volumes (mL) of ET, necrotic core, edema, tumor core (TC = ET+NCR) and whole tumor (WT); centroid; bounding-box extent; max axial diameter
* **Resection safety margins** — Euclidean distance map around the tumor core → 5/10/15 mm margin contours, and the extra brain tissue (mL) each margin would include
* **Entry-point trajectory check** — straight path from a chosen entry point to the tumor-core centroid: length, tissue crossed, edema/tumor crossed, distance to tumor
* **Interactive 3D** — surface meshes of each tumor sub-region inside a translucent brain shell, with optional margin shell

> Voxels are 1 mm isotropic in BraTS, so voxel distance = millimetres. The tool contains **no atlas of eloquent cortex / white-matter tracts / vessels**; a real planner would add these.

In [ ]:
from skimage import measure
import plotly.graph_objects as go

SPACING = 1.0   # mm (BraTS is 1 mm isotropic)


def tumor_report(seg, spacing=SPACING):
    vox = spacing ** 3 / 1000.0          # mL per voxel
    et, ncr, ed = (seg == 3), (seg == 1), (seg == 2)
    tc, wt = et | ncr, seg > 0
    rep = {"Enhancing tumor ET (mL)": et.sum() * vox, "Necrotic/non-enh. core (mL)": ncr.sum() * vox,
           "Peritumoral edema (mL)": ed.sum() * vox, "Tumor core TC (mL)": tc.sum() * vox,
           "Whole tumor WT (mL)": wt.sum() * vox}
    if wt.any():
        co = np.argwhere(wt)
        rep["Centroid (voxel x,y,z)"] = tuple(int(v) for v in co.mean(0).round())
        rep["Bounding box (mm)"] = tuple(int(v) for v in (co.max(0) - co.min(0) + 1) * spacing)
        diam = 0.0
        for z in np.unique(co[:, 2]):
            lab, n = ndi.label(wt[:, :, z])
            for rp in measure.regionprops(lab):
                ml = rp.axis_major_length if hasattr(rp, "axis_major_length") else rp.major_axis_length
                diam = max(diam, ml * spacing)
        rep["Max axial diameter (mm)"] = round(diam, 1)
        rep["Necrosis fraction of TC"] = float(ncr.sum() / max(tc.sum(), 1))
    return rep


def tc_distance(seg, spacing=SPACING):
    """Distance (mm) from every voxel to the tumor core (0 inside). None if no tumor core."""
    tgt = (seg == 1) | (seg == 3)
    return ndi.distance_transform_edt(~tgt, sampling=spacing).astype(np.float32) if tgt.any() else None


def margin_table(seg, brain, dist, margins=(5, 10, 15), spacing=SPACING):
    vox = spacing ** 3 / 1000.0
    tgt = (seg == 1) | (seg == 3)
    rows = []
    for m in margins:
        ring = (dist <= m) & ~tgt & brain
        rows.append({"Margin (mm)": m, "Extra tissue incl. in margin (mL)": round(ring.sum() * vox, 1),
                     "…of which already edema (mL)": round((ring & (seg == 2)).sum() * vox, 1)})
    return pd.DataFrame(rows)


def trajectory_report(entry_vox, seg, brain, dist, spacing=SPACING):
    tgt = np.argwhere((seg == 1) | (seg == 3))
    if len(tgt) == 0 or dist is None:
        return None
    c, e = tgt.mean(0), np.asarray(entry_vox, float)
    L = np.linalg.norm(c - e) * spacing
    n = max(int(L / spacing) + 1, 2)
    pts = np.linspace(e, c, n).round().astype(int).clip(0, np.array(seg.shape) - 1)
    labs = seg[pts[:, 0], pts[:, 1], pts[:, 2]]
    step = L / (n - 1)
    return {"Straight-line length to tumor-core centroid (mm)": round(L, 1),
            "Path through brain tissue (mm)": round(brain[pts[:, 0], pts[:, 1], pts[:, 2]].sum() * step, 1),
            "…through edema (mm)": round((labs == 2).sum() * step, 1),
            "…through tumor core (mm)": round(np.isin(labs, (1, 3)).sum() * step, 1),
            "Entry point → nearest tumor-core voxel (mm)": round(float(dist[tuple(e.round().astype(int))]), 1)}


def mesh_from_mask(mask, step=2, sigma=1.0):
    if mask.sum() < 50:
        return None
    m = np.pad(ndi.gaussian_filter(mask.astype(np.float32), sigma), 1)
    try:
        v, f, _, _ = measure.marching_cubes(m, level=0.5, step_size=step)
    except Exception:
        return None
    return v - 1, f


def make_3d_figure(seg, brain, dist=None, margin_mm=0, title="3D tumor model"):
    fig = go.Figure()

    def add(mask, name, color, opacity, step=2, sigma=1.0):
        mm = mesh_from_mask(mask, step, sigma)
        if mm is None:
            return
        v, f = mm
        fig.add_trace(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2], color=color,
                                opacity=opacity, name=name, showlegend=True, flatshading=False,
                                lighting=dict(ambient=0.55, diffuse=0.7, specular=0.2)))

    add(brain, "Brain", "#b0b0b0", 0.06, step=4, sigma=2.0)
    add(seg == 2, "Edema", "#26cc73", 0.22)
    add(seg == 1, "Necrotic / non-enh. core", "#eb2e2e", 0.95)
    add(seg == 3, "Enhancing tumor", "#ffd91a", 0.95)
    if margin_mm and dist is not None:
        add(dist <= margin_mm, f"{margin_mm} mm safety margin", "#00e5ff", 0.14, step=3, sigma=1.5)
    ax = dict(showbackground=False, showticklabels=False, title="", showgrid=False, zeroline=False)
    fig.update_layout(template="plotly_dark", title=title, scene=dict(xaxis=ax, yaxis=ax, zaxis=ax, aspectmode="data"),
                      margin=dict(l=0, r=0, t=40, b=0), height=620, legend=dict(bgcolor="rgba(0,0,0,0.3)"))
    return fig

In [ ]:
# ---- Demo of the planning tools on one validation patient (lightweight model's prediction) ----
cid = VAL_IDS[0]
img, gt = load_cached(cid)
pred, ms = predict(student, img)
brain = np.abs(img).sum(0) > 0
dist = tc_distance(pred)

print(f"=== {cid} | lightweight model inference: {ms:.0f} ms ===")
rep = tumor_report(pred)
for k, v in rep.items():
    print(f"{k:34s} {v if not isinstance(v, float) else round(v, 2)}")
print("\nSafety-margin analysis (around tumor core):")
display(margin_table(pred, brain, dist))

# Entry point: cortical surface directly above the tumor-core centroid
cx, cy, _ = np.argwhere((pred == 1) | (pred == 3)).mean(0).round().astype(int)
cz_top = np.where(brain[cx, cy])[0].max()
traj = trajectory_report((cx, cy, cz_top), pred, brain, dist)
print("\nTrajectory check (vertex entry point above tumor):")
for k, v in traj.items():
    print(f"  {k:48s} {v}")

# Safety-margin visualisation on 3 planes
fig, axs = plt.subplots(1, 3, figsize=(17, 5.6))
c = np.argwhere(pred > 0).mean(0).round().astype(int)
for a, (pl, ax_i) in zip(axs, AXES.items()):
    idx = c[ax_i]
    draw(a, take(img[2], ax_i, idx), take(pred, ax_i, idx), f"{pl} (slice {idx})", alpha=0.45,
         dist2d=take(dist, ax_i, idx), margins=(5, 10, 15))
from matplotlib.lines import Line2D
fig.legend(handles=legend_handles() + [Line2D([0], [0], color=c_, lw=2, label=f"{m} mm margin")
                                       for m, c_ in zip((5, 10, 15), ("cyan", "orange", "magenta"))],
           loc="lower center", ncol=6, frameon=False)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.savefig(CFG.FIG_DIR / "fig6_safety_margins.png", dpi=130); plt.show()

In [ ]:
# ---- Interactive 3D (opens inline; also saved as standalone HTML) ----
fig3d = make_3d_figure(pred, brain, dist, margin_mm=10, title=f"{cid} — lightweight model, 10 mm margin")
fig3d.write_html(CFG.OUT_DIR / "tumor_3d.html", include_plotlyjs="cdn")
fig3d.show()

## 9. Further speed-ups: FP16 + ONNX export

FP16 GPU inference is already included in the benchmark above. Here we export the lightweight model to **ONNX** and time it with **ONNX Runtime on CPU** — handy for deployment on machines without a GPU.

In [ ]:
import onnx, onnxruntime as ort

onnx_path = CFG.OUT_DIR / "neuroassist_student.onnx"
try:
    s_cpu = copy.deepcopy(student).cpu().eval()
    dummy = torch.randn(1, 4, 128, 128, 128)
    try:
        torch.onnx.export(s_cpu, dummy, str(onnx_path), input_names=["mri"], output_names=["logits"],
                          opset_version=17, dynamo=False)
    except TypeError:                               # older torch without the `dynamo` kwarg
        torch.onnx.export(s_cpu, dummy, str(onnx_path), input_names=["mri"], output_names=["logits"], opset_version=17)
    onnx.checker.check_model(str(onnx_path))
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    xin = dummy.numpy()
    sess.run(None, {"mri": xin})
    ts = []
    for _ in range(3):
        t0 = time.perf_counter(); out = sess.run(None, {"mri": xin})[0]; ts.append((time.perf_counter() - t0) * 1000)
    with torch.no_grad():
        ref = s_cpu(dummy).numpy()
    print(f"ONNX saved: {onnx_path} ({onnx_path.stat().st_size/2**20:.1f} MB)")
    print(f"ONNX Runtime CPU: {np.mean(ts):.0f} ms  vs  PyTorch CPU: {results.loc[stu_name, 'CPU (ms)']:.0f} ms")
    print(f"max |PyTorch - ONNX| logit difference: {np.abs(ref - out).max():.2e}")
except Exception as e:
    print("ONNX export/benchmark skipped:", repr(e))

# 🔬 Extensions
Everything below builds on the trained baseline/student. **Cheap inference-time items run by default** (flip-TTA, uncertainty maps, ONNX INT8, channels-last/`torch.compile`, atlas overlay); **training-heavy items are gated by flags** in `CFG` (`RUN_CV`, `RUN_FEATURE_KD`, `RUN_REGION_HEAD`, `RUN_DWSEP`, `RUN_MC_DROPOUT_STUDENT`, `LONG_RUN`).

Results of each experiment are collected in `EXT` / `VARIANTS` and end up in the generated README.

In [ ]:
EXT = {}          # name -> markdown table (goes to README)
VARIANTS = []     # one row per trained model variant


def df_to_md(df, floatfmt="{:.3f}"):
    cols = [df.index.name or "model"] + list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        out.append("| " + " | ".join([str(idx)] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                                    for v in row.values]) + " |")
    return "\n".join(out)


def record_variant(name, model, val=None):
    """Add a row (params, latency, Dice) to the variants table."""
    val = val or validate(model, val_loader)
    dev, half = (DEVICE, True) if DEVICE.type == "cuda" else (torch.device("cpu"), False)
    ms = bench(model, dev, half=half, warm=1, runs=3)[0]
    VARIANTS.append({"variant": name, "Params (M)": round(count_params(model) / 1e6, 2), "latency (ms)": round(ms, 1),
                     "Dice ET": round(val["ET"], 3), "Dice TC": round(val["TC"], 3), "Dice WT": round(val["WT"], 3),
                     "Dice mean": round(val["mean"], 3)})


record_variant("Baseline (teacher)", teacher)
record_variant("Lightweight + logit KD", student)
if student_scratch is not None:
    record_variant("Lightweight, no KD", student_scratch)
pd.DataFrame(VARIANTS)

## E1. Test-time augmentation (flip TTA)

**Idea:** average the predictions over mirrored copies of the input, undoing each flip on the output:

$$\bar p(x)=\frac{1}{|\mathcal F|}\sum_{f\in\mathcal F} f^{-1}\!\big(\mathrm{softmax}(m(f(x)))\big),\qquad \mathcal F=\text{all 8 flip combinations over the 3 axes}$$

**Why:** the network is not exactly flip-equivariant, so each flip gives a slightly different, partly independent error; averaging reduces variance (typically a small Dice gain and fewer stray voxels → better HD95) at the price of |F|× compute.

In [ ]:
import itertools
FLIP_SETS = [c for r in range(4) for c in itertools.combinations((0, 1, 2), r)]      # 8 combinations


@torch.no_grad()
def predict_proba(model, img, tta=False, flip_sets=None):
    """Return (mean softmax probs (4,H,W,D) float32, latency ms, std over flips of tumor probability (H,W,D))."""
    model.eval()
    x = torch.from_numpy(img).float().unsqueeze(0).to(DEVICE)
    sets = (flip_sets or FLIP_SETS) if tta else [()]
    amp = CFG.USE_AMP and DEVICE.type == "cuda"
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    s1 = s2 = tum1 = tum2 = None
    for fs in sets:
        dims = [d + 2 for d in fs]
        with torch.autocast(DEVICE.type, enabled=amp):
            lg = model(torch.flip(x, dims) if dims else x)
        p = F.softmax(lg.float(), 1)
        if dims:
            p = torch.flip(p, dims)
        tum = 1 - p[:, 0]
        s1 = p if s1 is None else s1 + p
        tum1 = tum if tum1 is None else tum1 + tum
        tum2 = tum ** 2 if tum2 is None else tum2 + tum ** 2
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    n = len(sets)
    std = ((tum2 / n - (tum1 / n) ** 2).clamp_min(0).sqrt())[0]
    return (s1 / n)[0].cpu().numpy(), ms, std.cpu().numpy()


def evaluate_tta(models, ids):
    rows = []
    for m in models.values():
        predict_proba(m, load_cached(ids[0])[0])
    for cid in tqdm(ids, desc="TTA eval"):
        img, gt = load_cached(cid)
        for name, m in models.items():
            for mode in ("no TTA", "flip TTA (8x)"):
                probs, ms, _ = predict_proba(m, img, tta=mode != "no TTA")
                pred = probs.argmax(0).astype(np.uint8)
                row = {"model": name, "mode": mode, "latency_ms": ms}
                for r, lab in REGIONS.items():
                    p, g = np.isin(pred, lab), np.isin(gt, lab)
                    row[f"Dice_{r}"], row[f"HD95_{r}"] = dice_np(p, g), hd95_np(p, g)
                rows.append(row)
    df = pd.DataFrame(rows)
    df["Dice_mean"] = df[["Dice_ET", "Dice_TC", "Dice_WT"]].mean(axis=1)
    return df


tta_df = evaluate_tta({"Baseline": teacher, "Lightweight": student}, VAL_IDS)
tta_sum = tta_df.groupby(["model", "mode"]).mean(numeric_only=True).round(3)
tta_sum

In [ ]:
# Effect of TTA (delta = TTA minus no-TTA)
cols = ["Dice_ET", "Dice_TC", "Dice_WT", "Dice_mean", "HD95_ET", "HD95_TC", "HD95_WT", "latency_ms"]
delta = (tta_sum.xs("flip TTA (8x)", level="mode") - tta_sum.xs("no TTA", level="mode"))[cols].round(3)
delta.columns = [c + " Δ" for c in cols]
display(delta)
EXT["Flip-TTA (8 flips): mean over validation cases"] = df_to_md(tta_sum.reset_index().assign(key=lambda d: d["model"] + " / " + d["mode"]).set_index("key").drop(columns=["model", "mode"])[cols], "{:.3f}")

## E2. Train longer and 5-fold cross-validation

* `CFG.LONG_RUN = True` trains 150 epochs per model (cosine schedule stretches automatically).
* `CFG.RUN_CV = True` runs **5-fold patient-level cross-validation** of the lightweight model and reports mean ± std. By default each fold trains the student **without** KD (using the single teacher would leak: it has seen other folds' validation cases). Set `CFG.CV_WITH_KD = True` to also train a fold-specific teacher (≈5× teacher cost).

In [ ]:
def run_cross_validation(epochs=None, with_kd=False):
    epochs = epochs or CFG.CV_EPOCHS
    ids = np.array(ALL_IDS)
    perm = np.random.RandomState(CFG.SEED).permutation(len(ids))
    folds = np.array_split(perm, CFG.N_FOLDS)
    rows = []
    for k in range(CFG.N_FOLDS):
        va_ids = sorted(ids[folds[k]].tolist())
        tr_ids = sorted(ids[np.concatenate([folds[j] for j in range(CFG.N_FOLDS) if j != k])].tolist())
        tl, vl = make_loaders(tr_ids, va_ids)
        t_f = None
        if with_kd:
            t_f, _ = train_model(f"cv{k}_teacher", build_teacher(), tl, vl, epochs)
        s_f, _ = train_model(f"cv{k}_student", build_student(), tl, vl, epochs, teacher=t_f,
                             kd_alpha=CFG.KD_ALPHA if with_kd else 0.0, kd_T=CFG.KD_TEMP)
        v = validate(s_f, vl)
        rows.append({"fold": k + 1, **{r: v[r] for r in REGIONS}, "mean": v["mean"]})
        print(f"fold {k+1}: {rows[-1]}")
        del tl, vl, t_f, s_f
    cv = pd.DataFrame(rows).set_index("fold")
    cv.loc["mean"] = cv.mean(); cv.loc["std"] = cv.iloc[:-1].std()
    return cv.round(3)


if CFG.RUN_CV:
    cv_table = run_cross_validation(with_kd=CFG.CV_WITH_KD)
    display(cv_table)
    EXT[f"{CFG.N_FOLDS}-fold CV, lightweight model ({'with' if CFG.CV_WITH_KD else 'without'} KD, {CFG.CV_EPOCHS} epochs/fold)"] = \
        df_to_md(cv_table, "{:.3f}")
else:
    print("Skipped (CFG.RUN_CV = False)")

## E3. Feature-level distillation

**Equation:**
$$\mathcal L_{feat}=\frac1L\sum_{l=1}^{L}\Big\|\;\mathrm{N}\big(A_l(f^{s}_l)\big)-\mathrm{N}\big(f^{t}_l\big)\Big\|_2^2,\qquad \mathrm N(f)=\frac{f}{\|f\|_{2,\text{channel}}}$$

where $f^{s}_l,f^{t}_l$ are encoder features of student/teacher at level $l$ (same spatial size because both nets share the topology), and $A_l$ is a learnable 1×1×1 conv mapping the student's channels (8…128) to the teacher's (32…320).

**Why:** logit-KD only supervises the output; matching intermediate features also teaches the student *what the encoder should represent*. The adapter fixes the channel mismatch; per-voxel channel-L2 normalisation makes the loss scale-invariant (student and teacher activations have different magnitudes). Total loss: $\mathcal L=\mathcal L_{seg}+\alpha T^2 KL+\lambda\,\mathcal L_{feat}$.

In [ ]:
class FeatureDistiller(nn.Module):
    """Forward hooks on the 5 encoder stages of student & teacher (DynUNet: input_block, downsamples, bottleneck)."""

    def __init__(self, student, teacher):
        super().__init__()
        enc = lambda m: [m.input_block, *m.downsamples, m.bottleneck]
        self.s_feats, self.t_feats, self.enabled, self.handles = {}, {}, False, []
        s_mods, t_mods = enc(student), enc(teacher)
        assert len(s_mods) == len(t_mods) == len(CFG.STUDENT_FILTERS)
        for i, (sm, tm) in enumerate(zip(s_mods, t_mods)):
            self.handles.append(sm.register_forward_hook(lambda m, a, o, i=i: self.s_feats.__setitem__(i, o) if self.enabled else None))
            self.handles.append(tm.register_forward_hook(lambda m, a, o, i=i: self.t_feats.__setitem__(i, o) if self.enabled else None))
        self.adapters = nn.ModuleList([nn.Conv3d(s, t, 1) for s, t in zip(CFG.STUDENT_FILTERS, CFG.TEACHER_FILTERS)])

    def loss(self):
        tot = 0.0
        for i, a in enumerate(self.adapters):
            fs = F.normalize(a(self.s_feats[i]).float(), dim=1)
            ft = F.normalize(self.t_feats[i].float().detach(), dim=1)
            tot = tot + F.mse_loss(fs, ft)
        self.s_feats.clear(); self.t_feats.clear()
        return tot / len(self.adapters)

    def close(self):
        for h in self.handles:
            h.remove()


student_fd = None
if CFG.RUN_FEATURE_KD:
    student_fd = build_student()
    fd = FeatureDistiller(student_fd, teacher).to(DEVICE)
    student_fd, hist_fd = train_model("student_featkd", student_fd, train_loader, val_loader, CFG.STUDENT_EPOCHS,
                                      teacher=teacher, kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP,
                                      feat_distill=fd, feat_lambda=CFG.FEAT_LAMBDA)
    fd.close()
    record_variant("Lightweight + logit KD + feature KD", student_fd)
else:
    print("Skipped (CFG.RUN_FEATURE_KD = False)")

## E4. Region-based sigmoid head (WT / TC / ET)

**Idea:** predict the three *nested, overlapping* BraTS regions with 3 independent sigmoid outputs instead of 4 mutually exclusive softmax classes:
$$\mathcal L=\underbrace{1-\tfrac{1}{3}\sum_{r}\tfrac{2\sum_i \sigma(z_{ir})\,y_{ir}}{\sum_i\sigma(z_{ir})+\sum_i y_{ir}}}_{\text{Dice}}\;+\;\underbrace{\text{BCE}(\sigma(z),y)}_{\text{per-region}},\qquad y_{WT}=\mathbb 1[l>0],\;y_{TC}=\mathbb 1[l\in\{1,3\}],\;y_{ET}=\mathbb 1[l=3]$$

**Why:** the challenge is *scored* on these regions, so the loss optimises exactly them; each region gets its own decision threshold; classes are not forced to compete (TC ⊂ WT, ET ⊂ TC). Decode with a hierarchy: ET→3, TC∖ET→1, WT∖TC→2.

In [ ]:
from monai.losses import DiceLoss


def regions_from_labels(y):                       # (B,1,...) long -> (B,3,...) float [WT, TC, ET]
    y = y[:, 0]
    return torch.stack([y > 0, (y == 1) | (y == 3), y == 3], 1).float()


def decode_regions(prob):                         # (B,3,...) sigmoid probs -> (B,...) label volume (0,1,2,3)
    wt, tc, et = prob[:, 0] > .5, prob[:, 1] > .5, prob[:, 2] > .5
    lab = torch.zeros_like(wt, dtype=torch.long)
    lab[wt] = 2
    lab[wt & tc] = 1
    lab[wt & tc & et] = 3
    return lab


def train_region_student(epochs):
    model = build_model(CFG.STUDENT_FILTERS, False, out_ch=3).to(DEVICE)
    dice_l, use_amp = DiceLoss(sigmoid=True, batch=True), CFG.USE_AMP and DEVICE.type == "cuda"
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=CFG.LR * 0.02)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=use_amp)
    best, best_sd = -1, None
    for ep in range(epochs):
        model.train()
        for b in tqdm(train_loader, leave=False, desc=f"region-head {ep+1}/{epochs}"):
            x, y = b["image"].to(DEVICE), regions_from_labels(b["label"].long().to(DEVICE))
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=use_amp):
                z = model(x).float()
                loss = dice_l(z, y) + F.binary_cross_entropy_with_logits(z, y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 12.0); scaler.step(opt); scaler.update()
        sched.step()
        if (ep + 1) % CFG.VAL_EVERY == 0 or ep + 1 == epochs:
            model.eval(); sc = {r: [] for r in REGIONS}
            with torch.no_grad():
                for b in val_loader:
                    with torch.autocast(DEVICE.type, enabled=use_amp):
                        z = model(b["image"].to(DEVICE))
                    pred = decode_regions(torch.sigmoid(z.float()))
                    d = region_dice_torch(pred[0], b["label"].long().to(DEVICE)[0, 0])
                    for r in REGIONS: sc[r].append(d[r])
            v = {r: float(np.mean(s)) for r, s in sc.items()}; v["mean"] = float(np.mean(list(v.values())))
            print(f"[region-head] ep {ep+1} Dice ET {v['ET']:.3f} TC {v['TC']:.3f} WT {v['WT']:.3f} mean {v['mean']:.3f}")
            if v["mean"] > best:
                best, best_v, best_sd = v["mean"], v, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_sd)
    return model, best_v


if CFG.RUN_REGION_HEAD:
    student_region, v_region = train_region_student(CFG.STUDENT_EPOCHS)
    record_variant("Lightweight, sigmoid region head (no KD)", student_region, v_region)
else:
    print("Skipped (CFG.RUN_REGION_HEAD = False)")

## E5. Further speed-ups

| Trick | Why it can help |
|---|---|
| **Depthwise-separable 3D convs** | a k³ conv costs $k^3 C_{in}C_{out}$ weights/MACs; depthwise + 1×1×1 costs $k^3C_{in}+C_{in}C_{out}$, a ratio of $\approx 1/C_{out}+1/k^3$ (≈ 1/27 for wide layers). *Caveat:* 3D depthwise kernels are memory-bound on GPUs, so **always benchmark** — it may shrink params/FLOPs without lowering latency |
| **channels-last (NDHWC) + `torch.compile`** | tensor-core friendly memory layout and fused kernels, no retraining |
| **INT8 static quantization (ONNX Runtime)** | 8-bit weights *and* activations $q=\mathrm{round}(x/s)+z$, with scales calibrated on real MRI patches; ~4× smaller, faster on CPUs with VNNI/AVX-512 |

In [ ]:
def to_depthwise_separable(model):
    """Replace every 3x3x3 Conv3d by depthwise(3x3x3, groups=C_in) + pointwise(1x1x1)."""
    for name, ch in list(model.named_children()):
        if isinstance(ch, nn.Conv3d) and ch.kernel_size == (3, 3, 3):
            dw = nn.Conv3d(ch.in_channels, ch.in_channels, 3, stride=ch.stride, padding=ch.padding,
                           groups=ch.in_channels, bias=False)
            pw = nn.Conv3d(ch.in_channels, ch.out_channels, 1, bias=ch.bias is not None)
            setattr(model, name, nn.Sequential(dw, pw))
        else:
            to_depthwise_separable(ch)
    return model


def build_student_dw():
    return to_depthwise_separable(build_model(CFG.STUDENT_DW_FILTERS, False))


m_dw = build_student_dw()
print(f"Depthwise-separable student: {count_params(m_dw)/1e6:.2f} M params (plain student: {count_params(student)/1e6:.2f} M)")
if CFG.RUN_DWSEP:
    student_dw, _ = train_model("student_dw", m_dw, train_loader, val_loader, CFG.STUDENT_EPOCHS, teacher=teacher,
                                kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP)
    record_variant("Depthwise-separable student + KD", student_dw)
else:
    print("Training skipped (CFG.RUN_DWSEP = False)")

In [ ]:
# ---- channels-last + torch.compile benchmark (GPU, FP16) ----
def _time(m, x, warm=3, runs=10):
    with torch.no_grad():
        for _ in range(warm): m(x)
        ts = []
        for _ in range(runs):
            torch.cuda.synchronize(); t0 = time.perf_counter(); m(x); torch.cuda.synchronize()
            ts.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(ts))


def bench_variants(model, shape=(1, 4, 128, 128, 128)):
    rows = []
    x = torch.randn(*shape, device=DEVICE).half()
    m0 = copy.deepcopy(model).to(DEVICE).eval().half()
    rows.append(("eager FP16", _time(m0, x)))
    m1 = copy.deepcopy(model).to(DEVICE).eval().half().to(memory_format=torch.channels_last_3d)
    x1 = x.contiguous(memory_format=torch.channels_last_3d)
    rows.append(("channels-last FP16", _time(m1, x1)))
    try:
        mc = torch.compile(m1)
        rows.append(("channels-last + torch.compile FP16", _time(mc, x1, warm=3)))
    except Exception as e:
        print("torch.compile unavailable:", repr(e)[:120])
    return pd.DataFrame(rows, columns=["variant", "latency (ms)"]).round(2)


if DEVICE.type == "cuda":
    speed_df = bench_variants(student)
    speed_df["speed-up vs eager"] = (speed_df["latency (ms)"].iloc[0] / speed_df["latency (ms)"]).round(2)
    display(speed_df)
    EXT["Lightweight model: channels-last / torch.compile (GPU FP16, 128³)"] = df_to_md(speed_df.set_index("variant"), "{:.2f}")
else:
    print("GPU not available - skipping channels-last / torch.compile benchmark")

In [ ]:
# ---- INT8 static quantization with ONNX Runtime (CPU) ----
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType


class _Calib(CalibrationDataReader):
    def __init__(self, ids):
        self.it = iter([{"mri": load_cached(c)[0][None]} for c in ids])

    def get_next(self):
        return next(self.it, None)


def ort_eval(path, ids):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    d, ts = {r: [] for r in REGIONS}, []
    for cid in ids:
        img, gt = load_cached(cid)
        t0 = time.perf_counter(); out = sess.run(None, {"mri": img[None]})[0]; ts.append((time.perf_counter() - t0) * 1000)
        pred = out[0].argmax(0)
        for r, lab in REGIONS.items():
            d[r].append(dice_np(np.isin(pred, lab), np.isin(gt, lab)))
    return {r: float(np.mean(v)) for r, v in d.items()}, float(np.mean(ts))


try:
    int8_path = CFG.OUT_DIR / "neuroassist_student_int8.onnx"
    quantize_static(str(onnx_path), str(int8_path), _Calib(TRAIN_IDS[:8]), quant_format=QuantFormat.QDQ,
                    per_channel=True, activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    eval_ids = VAL_IDS[:8]
    d32, t32 = ort_eval(onnx_path, eval_ids)
    d8, t8 = ort_eval(int8_path, eval_ids)
    q = pd.DataFrame([{"model": "ONNX FP32", "size (MB)": onnx_path.stat().st_size / 2**20, "CPU ms": t32, **d32},
                      {"model": "ONNX INT8 (static, QDQ)", "size (MB)": int8_path.stat().st_size / 2**20, "CPU ms": t8, **d8}]
                     ).set_index("model").round(3)
    display(q)
    EXT["INT8 static quantization (ONNX Runtime, CPU, first 8 val cases)"] = df_to_md(q, "{:.3f}")
except Exception as e:
    print("INT8 quantization skipped:", repr(e)[:300])

## E6. Uncertainty — flag voxels a surgeon should double-check

Three complementary maps (all per-voxel):

1. **Teacher–student disagreement** (Jensen–Shannon divergence between the two softmax outputs; no retraining)
$$\mathrm{JS}(p^t,p^s)=\tfrac12\mathrm{KL}(p^t\,\|\,m)+\tfrac12\mathrm{KL}(p^s\,\|\,m),\qquad m=\tfrac12(p^t+p^s)$$
   *Why JS:* symmetric, bounded by ln 2 and finite even if a probability is 0 (plain KL is neither).
2. **Flip-TTA predictive entropy** $H(\bar p)=-\sum_c\bar p_c\log\bar p_c$ and the **std over flips** of the tumor probability.
3. **MC-dropout** (needs a student trained with dropout: `CFG.RUN_MC_DROPOUT_STUDENT=True`): keep dropout active at test time, sample $N$ passes, use entropy/variance of $\frac1N\sum_n p_n$.

We check usefulness quantitatively: do the **top 5 % most uncertain voxels** around the tumor contain a much larger share of the model's actual errors than the 5 % expected by chance?

In [ ]:
def js_divergence_map(p, q, eps=1e-8):
    m = 0.5 * (p + q)
    kl = lambda a, b: (a * (np.log(a + eps) - np.log(b + eps))).sum(0)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def entropy_map(p, eps=1e-8):
    return -(p * np.log(p + eps)).sum(0)


def enable_mc_dropout(model):
    model.eval(); n = 0
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
            m.train(); n += 1
    return n


@torch.no_grad()
def mc_dropout_maps(model, img, n=None):
    n = n or CFG.MC_SAMPLES
    if enable_mc_dropout(model) == 0:
        raise RuntimeError("model has no dropout layers (train with CFG.STUDENT_DROPOUT > 0)")
    x = torch.from_numpy(img).float().unsqueeze(0).to(DEVICE)
    ps = []
    for _ in range(n):
        with torch.autocast(DEVICE.type, enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
            ps.append(F.softmax(model(x).float(), 1)[0].cpu().numpy())
    model.eval()
    ps = np.stack(ps); mean = ps.mean(0)
    return mean, entropy_map(mean), (1 - ps[:, 0]).std(0)


def uncertainty_summary(unc, err, roi, top=0.05):
    thr = np.percentile(unc[roi], 100 * (1 - top))
    flagged = roi & (unc >= thr)
    return {"errors in ROI (vox)": int((err & roi).sum()),
            f"share of errors inside top-{int(top*100)}% uncertain voxels": float((err & flagged).sum() / max((err & roi).sum(), 1)),
            "mean unc. at errors / elsewhere": float(unc[err & roi].mean() / (unc[roi & ~err].mean() + 1e-8))}


rows, cache = [], {}
for cid in VAL_IDS[:12]:
    img, gt = load_cached(cid)
    p_t, _, _ = predict_proba(teacher, img)
    p_s, _, tta_std = predict_proba(student, img, tta=True)
    pred = p_s.argmax(0).astype(np.uint8)
    unc = {"Teacher–student JS": js_divergence_map(p_t, p_s), "Flip-TTA entropy": entropy_map(p_s), "Flip-TTA std": tta_std}
    roi = ndi.binary_dilation((gt > 0) | (pred > 0), iterations=8)          # 8 mm band around the tumor
    err = pred != gt
    for name, u in unc.items():
        rows.append({"case": cid, "map": name, **uncertainty_summary(u, err, roi)})
    cache[cid] = (img, gt, pred, unc, roi, err)
unc_df = pd.DataFrame(rows).groupby("map").mean(numeric_only=True).round(3)
display(unc_df)
EXT["Uncertainty maps: do the top-5% uncertain voxels capture the errors? (chance = 0.05)"] = df_to_md(unc_df, "{:.3f}")

In [ ]:
# ---- Uncertainty figure ----
cid = VAL_IDS[0]; img, gt, pred, unc, roi, err = cache[cid]
k = best_slice(pred if (pred > 0).any() else gt, 2); base = take(img[2], 2, k)
fig, axs = plt.subplots(1, 4, figsize=(20, 4.8))
draw(axs[0], base, take(pred, 2, k), f"{cid}: lightweight prediction")
for a, name in zip(axs[1:3], ["Teacher–student JS", "Flip-TTA entropy"]):
    u = unc[name].copy(); u[~roi] = 0
    im = a.imshow(np.ma.masked_less_equal(take(u, 2, k), 0).T, origin="lower", cmap="magma")
    a.imshow(np.transpose(np.stack([gray_norm(base)] * 3, -1), (1, 0, 2)), origin="lower", alpha=0.35)
    a.set_title(name); a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
flag = (unc["Teacher–student JS"] >= np.percentile(unc["Teacher–student JS"][roi], 95)) & roi
rgb = np.transpose(np.stack([gray_norm(base)] * 3, -1), (1, 0, 2)).copy()
rgb[take(flag, 2, k).T] = [0, 0.9, 1]; rgb[take(err, 2, k).T & ~take(flag, 2, k).T] = [1, 0, 1]
axs[3].imshow(rgb, origin="lower"); axs[3].set_title("cyan = 'double-check' voxels · magenta = other errors"); axs[3].axis("off")
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / "fig7_uncertainty.png", dpi=130); plt.show()

In [ ]:
# ---- (optional) MC-dropout student ----
student_mc = None
if CFG.RUN_MC_DROPOUT_STUDENT:
    student_mc = build_model(CFG.STUDENT_FILTERS, False, dropout=CFG.STUDENT_DROPOUT)
    student_mc, _ = train_model("student_mc", student_mc, train_loader, val_loader, CFG.STUDENT_EPOCHS, teacher=teacher,
                                kd_alpha=CFG.KD_ALPHA, kd_T=CFG.KD_TEMP)
    record_variant(f"Lightweight + KD + dropout {CFG.STUDENT_DROPOUT} (MC-dropout)", student_mc)
    img, gt = load_cached(VAL_IDS[0])
    mean_p, ent, std = mc_dropout_maps(student_mc, img)
    print(f"MC-dropout ({CFG.MC_SAMPLES} samples): mean entropy {ent.mean():.4f}, max std {std.max():.3f}")
else:
    print("Skipped (CFG.RUN_MC_DROPOUT_STUDENT = False)")

## E7. Eloquent-cortex overlay (Harvard-Oxford atlas)

For real margin planning the *function* of the tissue next to the tumor matters. This cell resamples the **Harvard-Oxford cortical atlas** into the patient grid and reports, for functionally critical areas (motor/sensory strips, Broca/Wernicke-type language areas, visual cortex), the **minimum distance from the tumor core**:
$$d_R=\min_{x\in R}D(x),\qquad D(x)=\min_{t\in TC}\|x-t\|_2$$
and flags any region with $d_R\le m$ (the chosen safety margin).

> ⚠️ **Approximation:** BraTS is co-registered to the SRI24 template, which is close to but **not identical** with MNI152. A proper pipeline would non-linearly register each case to MNI (e.g. ANTs SyN) and add white-matter tracts (HCP tractography atlas). Requires internet for the atlas download (`nilearn`).

In [ ]:
ELOQUENT = {"Precentral": "motor", "Postcentral": "sensory", "opercularis": "language (Broca area)",
            "Superior Temporal Gyrus, posterior": "language (Wernicke area)", "Supramarginal Gyrus, posterior": "language",
            "Angular": "language/reading", "Heschl": "auditory", "Occipital Pole": "visual", "Lateral Occipital Cortex, superior": "visual"}


def load_atlas_in_case_space(cid):
    from nilearn import datasets, image
    atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-1mm")
    ref = nib.load(find_case_files(os.path.join(DATA_ROOT, cid))["flair"])
    try:
        res = image.resample_to_img(atlas.maps, ref, interpolation="nearest", force_resample=True, copy_header=True)
    except TypeError:
        res = image.resample_to_img(atlas.maps, ref, interpolation="nearest")
    return np.asarray(res.dataobj).astype(np.int16)[CFG.CROP], list(atlas.labels)


def eloquent_report(cid, seg, margin_mm=10):
    dist = tc_distance(seg)
    if dist is None:
        return None, None, None
    vol, labels = load_atlas_in_case_space(cid)
    rows = []
    for i, name in enumerate(labels):
        func = next((f for key, f in ELOQUENT.items() if key in name), None)
        if func is None or (vol == i).sum() == 0:
            continue
        d = float(dist[vol == i].min())
        rows.append({"region": name, "function": func, "min distance to tumor core (mm)": round(d, 1),
                     f"within {margin_mm} mm margin": "YES" if d <= margin_mm else "no", "_idx": i})
    return pd.DataFrame(rows).sort_values("min distance to tumor core (mm)"), vol, dist


try:
    cid = VAL_IDS[0]
    img, gt = load_cached(cid); pred, _ = predict(student, img)
    rep_el, atlas_vol, dist_el = eloquent_report(cid, pred, margin_mm=10)
    display(rep_el.drop(columns="_idx"))
    cen = np.argwhere((pred == 1) | (pred == 3)).mean(0).round().astype(int)
    fig, axs = plt.subplots(1, 3, figsize=(17, 5.6)); cols_ = plt.cm.tab10(np.linspace(0, 1, 10))
    for a, (pl, ax_i) in zip(axs, AXES.items()):
        idx = cen[ax_i]; draw(a, take(img[2], ax_i, idx), take(pred, ax_i, idx), f"{pl} (slice {idx})", alpha=0.5,
                              dist2d=take(dist_el, ax_i, idx), margins=(10,))
        for j, (_, r) in enumerate(rep_el.head(6).iterrows()):
            m = take(atlas_vol, ax_i, idx) == r["_idx"]
            if m.any():
                a.contour(m.T.astype(float), levels=[0.5], colors=[cols_[j]], linewidths=1.6)
    from matplotlib.lines import Line2D
    fig.legend(handles=[Line2D([0], [0], color=cols_[j], lw=2, label=f"{r['region'][:32]} ({r['min distance to tumor core (mm)']} mm)")
                        for j, (_, r) in enumerate(rep_el.head(6).iterrows())], loc="lower center", ncol=3, frameon=False)
    plt.tight_layout(rect=(0, 0.1, 1, 1)); plt.savefig(CFG.FIG_DIR / "fig8_eloquent_atlas.png", dpi=130); plt.show()
    EXT["Eloquent-cortex proximity (approximate atlas overlay), one validation case"] = df_to_md(
        rep_el.drop(columns="_idx").set_index("region"), "{:.1f}")
except Exception as e:
    print("Atlas overlay skipped (needs internet + nilearn):", repr(e)[:200])

In [ ]:
# ---- Variants comparison (everything trained so far) ----
variants_df = pd.DataFrame(VARIANTS).set_index("variant")
display(variants_df)
EXT["Model variants (validation Dice; latency = GPU FP16 or CPU, 128³)"] = df_to_md(variants_df, "{:.3f}")
variants_df.to_csv(CFG.OUT_DIR / "variants.csv")

## 10. Interactive clinical-style demo (Gradio)

Features: pick a held-out patient **or upload the 4 NIfTI sequences** (BraTS space, 240×240×155, file names containing `flair`, `t1`, `t1ce`, `t2`) · all 4 MRI modalities · segmentation overlay in axial/coronal/sagittal planes · **side-by-side baseline vs lightweight** (+ ground truth when available) · adjustable **safety-margin** contour · interactive **3D** tumor · **surgical report** (volumes, extent, margin tissue, latency, model agreement).

The cell launches a public share link (`share=True`) so you can open it on another device / show judges. Run the cell and open the printed URL.

In [ ]:
import gradio as gr
from matplotlib.figure import Figure
from matplotlib.patches import Patch

# Make sure we use the best checkpoints, and warm the models up once so reported latency is realistic
for _name, _m in (("teacher", teacher), ("student_kd", student)):
    _p = CFG.CKPT_DIR / f"{_name}_best.pt"
    if _p.exists():
        _m.load_state_dict(torch.load(_p, map_location=DEVICE, weights_only=False)["model"])
    _m.to(DEVICE).eval()
_ = [predict(m, load_cached(VAL_IDS[0])[0]) for m in (teacher, student)]

SEG_CHOICES = ["Side-by-side (baseline vs lightweight)", "Lightweight (student)", "Baseline (teacher)", "Ground truth"]
KEY_OF = {SEG_CHOICES[1]: "ps", SEG_CHOICES[2]: "pt", SEG_CHOICES[3]: "gt"}
MOD_TITLES = {"flair": "T2-FLAIR", "t1": "T1", "t1ce": "T1 post-contrast", "t2": "T2"}


def _path(f):
    return f if isinstance(f, str) else getattr(f, "name", str(f))


def load_upload(files):
    found = {}
    for f in files or []:
        p = _path(f)
        m = re.search(r"(flair|t1ce|t1gd|t1|t2)\.nii", os.path.basename(p).lower())
        if m:
            found["t1ce" if m.group(1) == "t1gd" else m.group(1)] = p
    missing = [k for k in CFG.MODALITIES if k not in found]
    if missing:
        raise gr.Error(f"Missing modalities in upload: {missing}. Upload 4 files whose names end in "
                       f"_flair / _t1 / _t1ce / _t2 (.nii or .nii.gz).")
    vols = {k: load_nii(found[k]) for k in CFG.MODALITIES}
    if any(v.shape != (240, 240, 155) for v in vols.values()):
        raise gr.Error("Volumes must be BraTS-space (240x240x155). Please register/resample first.")
    return preprocess_modalities(vols)


def make_state(img, gt, name):
    pt, t_ms = predict(teacher, img)
    ps, s_ms = predict(student, img)
    st = dict(name=name, img=img, gt=gt, pt=pt, ps=ps, t_ms=t_ms, s_ms=s_ms, brain=np.abs(img).sum(0) > 0, dist={})
    for k in ("pt", "ps", "gt"):
        if st[k] is not None:
            st["dist"][k] = tc_distance(st[k])
    return st


def build_report(st):
    L = [f"### Pre-operative summary — {st['name']}",
         "*Research prototype · not for clinical use · analysis in the 128³ brain-centred crop (1 mm voxels)*", ""]
    rows = [("Lightweight", "ps"), ("Baseline", "pt")] + ([("Ground truth", "gt")] if st["gt"] is not None else [])
    keys = ["Enhancing tumor ET (mL)", "Necrotic/non-enh. core (mL)", "Peritumoral edema (mL)",
            "Tumor core TC (mL)", "Whole tumor WT (mL)"]
    reps = {n: tumor_report(st[k]) for n, k in rows}
    L.append("| Volume (mL) | " + " | ".join(n for n, _ in rows) + " |")
    L.append("|---|" + "---|" * len(rows))
    for k in keys:
        L.append(f"| {k.replace(' (mL)', '')} | " + " | ".join(f"{reps[n][k]:.1f}" for n, _ in rows) + " |")
    r = reps["Lightweight"]
    if "Centroid (voxel x,y,z)" in r:
        L += ["", f"**Extent (lightweight):** bounding box {r['Bounding box (mm)']} mm · max axial diameter "
                  f"{r['Max axial diameter (mm)']} mm · necrosis fraction of core {r['Necrosis fraction of TC']:.0%} · "
                  f"centroid voxel {r['Centroid (voxel x,y,z)']}"]
        mt = margin_table(st["ps"], st["brain"], st["dist"]["ps"])
        L += ["", "**Resection safety margin around tumor core (lightweight model)**", "",
              "| Margin (mm) | Extra tissue included (mL) | …of which edema (mL) |", "|---|---|---|"]
        L += [f"| {int(a)} | {b} | {c} |" for a, b, c in mt.values]
    else:
        L += ["", "_No tumor detected by the lightweight model._"]
    agree = dice_np(st["pt"] > 0, st["ps"] > 0)
    L += ["", f"**Inference latency (128³ volume, {DEVICE.type.upper()}):** lightweight **{st['s_ms']:.0f} ms** vs baseline "
              f"**{st['t_ms']:.0f} ms** → {st['t_ms']/max(st['s_ms'],1e-6):.1f}× faster · whole-tumor agreement between models "
              f"(Dice) **{agree:.3f}**"]
    if st["gt"] is not None:
        L.append(f"\n**Accuracy vs ground truth (WT Dice):** lightweight {dice_np(st['ps']>0, st['gt']>0):.3f} · "
                 f"baseline {dice_np(st['pt']>0, st['gt']>0):.3f}")
    return "\n".join(L)


def run_case(case_id, files):
    if files:
        img, gt, name = load_upload(files), None, "Uploaded case"
    else:
        img, gt = load_cached(case_id); name = case_id
    st = make_state(img, gt, name)
    ref = st["ps"] if (st["ps"] > 0).any() else st["pt"]
    return st, build_report(st), gr.update(value=best_slice(ref, 2)), gr.update(value="Axial")


def _panel(ax, st, mi, plane_axis, idx, key, title, margin_mm):
    seg = None if key is None else st[key]
    dist = None if key is None else st["dist"].get(key)
    base = take(st["img"][mi], plane_axis, idx)
    draw(ax, base, None if seg is None else take(seg, plane_axis, idx), title,
         dist2d=None if dist is None else take(dist, plane_axis, idx), margins=(margin_mm,) if margin_mm else ())


def render(st, plane, idx, modality, seg_choice, margin_mm):
    if st is None:
        return None, None
    ax_i, mi, idx = AXES[plane], CFG.MODALITIES.index(modality), int(idx)
    if seg_choice == SEG_CHOICES[0]:
        panels = [(None, f"{MOD_TITLES[modality]} (raw)"), ("pt", "Baseline (teacher)"), ("ps", "Lightweight (student)")]
        if st["gt"] is not None:
            panels.append(("gt", "Ground truth"))
    else:
        panels = [(None, f"{MOD_TITLES[modality]} (raw)"), (KEY_OF[seg_choice], seg_choice)]
    fig = Figure(figsize=(4.6 * len(panels), 5.2), facecolor="#0e1117")
    for a, (key, title) in zip(fig.subplots(1, len(panels)), panels):
        _panel(a, st, mi, ax_i, idx, key, title, margin_mm)
        a.title.set_color("white")
    fig.legend(handles=[Patch(color=LABEL_COLORS[c], label=LABEL_NAMES[c]) for c in (3, 1, 2)], loc="lower center",
               ncol=3, frameon=False, labelcolor="white")
    if margin_mm:
        fig.text(0.99, 0.01, f"cyan contour = {margin_mm} mm margin around tumor core", color="cyan", ha="right", fontsize=9)
    fig.tight_layout(rect=(0, 0.06, 1, 1))

    fm = Figure(figsize=(18, 4.6), facecolor="#0e1117")
    ref = st["ps"]
    for a, (m, i) in zip(fm.subplots(1, 4), zip(CFG.MODALITIES, range(4))):
        draw(a, take(st["img"][i], ax_i, idx), take(ref, ax_i, idx), MOD_TITLES[m], alpha=0.35); a.title.set_color("white")
    fm.tight_layout()
    return fig, fm


def make_3d(st, seg_choice, margin_mm):
    if st is None:
        return None
    key = KEY_OF.get(seg_choice, "ps")
    if st.get(key) is None:          # e.g. "Ground truth" for an uploaded case
        key = "ps"
    return make_3d_figure(st[key], st["brain"], st["dist"].get(key), margin_mm, f"{st['name']} — {seg_choice}")


CSS = """
.gradio-container {max-width: 1500px !important}
#hdr {background: linear-gradient(90deg,#0b3d4a,#0e1117); padding: 14px 20px; border-radius: 10px; color: #e8f6f8}
#hdr h1 {margin: 0; font-size: 26px} #hdr p {margin: 4px 0 0 0; opacity: .85}
#warn {background:#3b2a05; color:#ffd98a; padding:8px 14px; border-radius:8px; font-size:13px}
"""
HEADER = ("<div id='hdr'><h1>🧠 NeuroAssist — Pre-operative Brain Tumor Planning</h1>"
          "<p>Real-time multi-sequence MRI segmentation · lightweight distilled 3D network · resection safety margins</p></div>")
WARN = "<div id='warn'>⚠️ Research prototype for demonstration only. Not a medical device; not validated for clinical decisions.</div>"

try:
    _blocks = gr.Blocks(theme=gr.themes.Soft(primary_hue="teal"), css=CSS, title="NeuroAssist")
except TypeError:                    # newer Gradio versions moved theme/css out of the constructor
    _blocks = gr.Blocks(title="NeuroAssist")

with _blocks as demo:
    gr.HTML(HEADER); gr.HTML(WARN)
    state = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            case_dd = gr.Dropdown(VAL_IDS, value=VAL_IDS[0], label="Held-out patient case")
            up = gr.File(label="…or upload 4 NIfTI sequences (flair / t1 / t1ce / t2)", file_count="multiple",
                         file_types=[".nii", ".gz"])
            run_btn = gr.Button("▶ Run segmentation (both models)", variant="primary")
            gr.Markdown("---")
            plane = gr.Radio(list(AXES), value="Axial", label="Plane")
            sl = gr.Slider(0, 127, value=64, step=1, label="Slice")
            modality = gr.Dropdown(CFG.MODALITIES, value="t1ce", label="Background MRI sequence")
            seg_choice = gr.Radio(SEG_CHOICES, value=SEG_CHOICES[0], label="Segmentation shown")
            margin = gr.Slider(0, 20, value=10, step=1, label="Safety margin around tumor core (mm)")
        with gr.Column(scale=3):
            with gr.Tabs():
                with gr.Tab("Segmentation viewer"):
                    main_plot = gr.Plot(label="")
                with gr.Tab("4 MRI modalities"):
                    mod_plot = gr.Plot(label="")
                with gr.Tab("3D tumor"):
                    b3d = gr.Button("Generate / refresh 3D model")
                    plot3d = gr.Plot(label="")
                with gr.Tab("Surgical report"):
                    report = gr.Markdown("Run a case to see the report.")
    ctrl = [state, plane, sl, modality, seg_choice, margin]
    run_btn.click(run_case, [case_dd, up], [state, report, sl, plane]).then(render, ctrl, [main_plot, mod_plot])
    for c in (plane, sl, modality, seg_choice, margin):
        c.change(render, ctrl, [main_plot, mod_plot])
    b3d.click(make_3d, [state, seg_choice, margin], plot3d)

try:
    demo.queue().launch(share=True, debug=False)
except Exception as e:               # no internet for the share tunnel -> local only
    print("share link unavailable:", repr(e))
    demo.queue().launch(share=False, debug=False)

## 10b. Export a bundle for the Streamlit app

Creates `streamlit_bundle/` (checkpoints, `config.json` with the model widths, `results_table.csv`, 10 held-out cases as `.npy`) and a `neuroassist_bundle.zip`. Put `app.py` (from the `streamlit_app/` folder) next to the unzipped bundle, or point the app's "Model bundle folder" field to it, then run `streamlit run app.py`.

In [ ]:
import shutil
BUNDLE = CFG.OUT_DIR / "streamlit_bundle"
(BUNDLE / "cases").mkdir(parents=True, exist_ok=True)
shutil.copy(CFG.CKPT_DIR / "teacher_best.pt", BUNDLE / "teacher_best.pt")
shutil.copy(CFG.CKPT_DIR / "student_kd_best.pt", BUNDLE / "student_kd_best.pt")
if (CFG.OUT_DIR / "results_table.csv").exists():
    shutil.copy(CFG.OUT_DIR / "results_table.csv", BUNDLE / "results_table.csv")
json.dump({"teacher_filters": list(CFG.TEACHER_FILTERS), "student_filters": list(CFG.STUDENT_FILTERS)},
          open(BUNDLE / "config.json", "w"))
for _cid in VAL_IDS[:10]:
    for _suf in ("img", "lbl"):
        shutil.copy(CFG.CACHE_DIR / f"{_cid}_{_suf}.npy", BUNDLE / "cases" / f"{_cid}_{_suf}.npy")
zip_path = shutil.make_archive(str(CFG.OUT_DIR / "neuroassist_bundle"), "zip", BUNDLE)
print("Bundle:", BUNDLE, "\nZip:", zip_path, f"({Path(zip_path).stat().st_size/2**20:.0f} MB)")

## 11. Auto-generate `README.md` (with the final results table) and `requirements.txt`

In [ ]:
def df_to_md(df, floatfmt="{:.3f}"):
    cols = [df.index.name or "model"] + list(df.columns)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        out.append("| " + " | ".join([str(idx)] + [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                                    for v in row.values]) + " |")
    return "\n".join(out)


eff_md = df_to_md(results[["Params (M)", "Size FP32 (MB)", "GFLOPs (128^3)", "GPU FP32 (ms)", "GPU FP16 (ms)", "CPU (ms)"]], "{:.2f}")
acc_md = df_to_md(results[["Dice_ET", "Dice_TC", "Dice_WT", "Dice_mean", "HD95_ET", "HD95_TC", "HD95_WT"]], "{:.3f}")

EXT_MD = "\n\n".join(f"### {k}\n{v}" for k, v in EXT.items()) or "_Extension experiments were not run._"

README = f"""# 🧠 NeuroAssist — Lightweight AI Surgical Planning Assistant for Brain Tumor Resection

*HackPrinceton project · BraTS2020 · PyTorch + MONAI*

## Problem
State-of-the-art brain-tumor segmentation models (nnU-Net, SwinUNETR, …) reach excellent Dice on BraTS but are
**large, slow and hard to deploy** on the machines that surgeons actually use for pre-operative planning or in the
operating room. Interactive planning (adjusting margins, comparing trajectories, re-running after re-scans) needs
**near-instant** inference on modest hardware.

## Solution
1. A strong **baseline** 3D nnU-Net-style network (MONAI `DynUNet`, residual blocks, filters {CFG.TEACHER_FILTERS}).
2. A **lightweight student** (filters {CFG.STUDENT_FILTERS}) trained with **knowledge distillation** (Dice + CE + KL to the teacher).
3. A **surgical-planning assistant**: tumor volumes, resection safety-margin maps, entry-point trajectory check,
   interactive 3D model and side-by-side baseline-vs-lightweight comparison in a Gradio UI.

**Headline:** {speed['Parameter reduction']:.1f}× fewer parameters, {speed['FLOP reduction']:.1f}× fewer FLOPs,
{speed['CPU speed-up']:.1f}× faster on CPU ({speed['GPU FP16 speed-up']:.1f}× on GPU FP16), with a mean-Dice change of
{-speed['Mean-Dice drop (abs.)']:+.3f} (ET/TC/WT).

## Results (held-out validation patients: {len(VAL_IDS)} of {len(ALL_IDS)}; patient-level split, seed {CFG.SEED})
### Efficiency (input 1×4×128×128×128)
{eff_md}

### Accuracy (mean over validation cases; HD95 in mm)
{acc_md}

{EXT_MD}

Figures are in `figures/` (training curves, efficiency plots, qualitative comparison, safety margins).

## Pipeline
* Per-modality z-score on brain voxels (0.5–99.5 percentile clip) → brain-centred crop 128³ → label remap **4→3**
* Training: 96³ foreground-biased patches; flips, rot90, affine, elastic, intensity shift/scale, noise
* Loss: Dice + CE (+ T²·KL distillation for the student); AdamW, cosine LR, AMP, gradient clipping
* Metrics: Dice (ET/TC/WT + NCR/ED), HD95, params, GFLOPs, latency (GPU FP32/FP16, CPU), VRAM
* ONNX export of the student + ONNX-Runtime CPU benchmark

## How to run
```bash
pip install monai nibabel scikit-image plotly gradio onnx onnxruntime tqdm joblib pandas
# open NeuroAssist_BraTS2020.ipynb (Kaggle: add dataset awsaf49/brats20-dataset-training-validation, enable GPU) and Run All
```
Set `CFG.QUICK_MODE = True` for a fast sanity run. The last notebook sections launch the Gradio demo.

## Limitations & disclaimer
Research prototype; **not a medical device**. Evaluated on a split of BraTS2020 training data only (no external
validation). No eloquent-cortex / tract / vessel atlas — margins are purely geometric. HD95 cases where a region is
empty on only one side are excluded.
"""
(CFG.OUT_DIR / "README.md").write_text(README)
(CFG.OUT_DIR / "requirements.txt").write_text("torch\nmonai>=1.3\nnibabel\nscikit-image\nscipy\nplotly\ngradio\n"
                                              "onnx\nonnxruntime\ntqdm\njoblib\npandas\nmatplotlib\n")
print(README[:1500]); print("\nSaved to", CFG.OUT_DIR)
for p in sorted(list(CFG.OUT_DIR.glob("*")) + list(CFG.FIG_DIR.glob("*")) + list(CFG.CKPT_DIR.glob("*_best.pt"))):
    print(f"  {p}  ({p.stat().st_size/2**20:.1f} MB)")

## 12. What's left / next steps
* Run the heavy extension flags on a full GPU session (`LONG_RUN`, `RUN_CV`, `RUN_FEATURE_KD`, `RUN_REGION_HEAD`, `RUN_DWSEP`, `RUN_MC_DROPOUT_STUDENT`) and compare in the variants table
* Replace the approximate atlas overlay by a proper **ANTs SyN registration** of each case to MNI space and add white-matter tracts (HCP tractography atlas)
* External validation (BraTS 2021 / institutional data) and clinician review of the safety-margin and uncertainty outputs